## action def

This notebook introduces `action def`; after running it you can declare a named action with typed inputs and outputs and an ordered sequence of sub-actions.

The Chapter 3 model expresses *what* the toaster must accomplish (the requirement) and *how much* energy it delivers (the calculation). Chapter 4 adds the functional layer: *how* the system transforms inputs into outputs step by step. `action def` in SysML v2 declares a named behavior with `in`/`out` parameters, a `first`/`then` sequence, and nested `action` steps. This notebook adds `ApplyHeat` to the model.

In [ ]:
from pathlib import Path
import opensysml
from toaster.report import format_diagnostics

conn = opensysml.connect(version="v0.9.0")
source = Path("../../models/ch04-cumulative.sysml").read_text()
print(source)
model = conn.load_from_content(source, strict=False)
assert model.ok, f"Model failed: {format_diagnostics(model.diagnostics)}"

The `ch04-cumulative.sysml` file adds `action def ApplyHeat` with sequential steps (`first start; then calculate; then done`) and a nested `calculate` action that calls `DeliveredEnergy`. Three `item def` types — `Start`, `Finish`, `Cancel` — represent items flowing between action steps. This is the functional decomposition of the heating operation: inputs, process, and outputs all named.

In [ ]:
# Negative control: an action def that references an undefined calculation
# in an assign statement raises "unresolved reference" at that site.
bad_source = """
package Bad {
    private import ScalarValues::*;
    action def BadAction {
        in power : Real;
        out energy : Real;
        first start;
        then action step { assign energy := UndefinedCalc(power); }
        then done;
    }
}
"""
bad = conn.load_from_content(bad_source, strict=False)
assert not bad.ok
print("Expected error:", bad.diagnostics[0].message)

In [ ]:
action = model.find("ToasterDemo::ApplyHeat")
assert action is not None
print(f"action kind: {action.kind}")
print(f"action id  : {action.id}")
conn.close()

`action def ApplyHeat { in power : Real; ... first start; then action calculate { ... } then done; }` is the A-F declaration; OpenSysML parses the sequence and sub-action assignments (O-S); `model.find()` returns the ActionDefinition symbol (E).

Try the chapter exercise in `exercises/ch04/exercise.ipynb`: declare a `Brew` action def for your coffee maker with `waterTemp` and `duration` inputs, a `first`/`then` sequence, and an assign step using `HeatRate`.